# Lab 04 — Modelando o Olist: do staging ao star schema (DuckDB)

**Onde roda:** 🟢 Browser (JupyterLite). Amostra pequena no estilo Olist (roda instantâneo).
Para o dataset completo, veja `datasets/README.md` e rode na bancada Docker.

Objetivo: aplicar os 4 passos de Kimball — construir dimensões com surrogate keys e a fato no grão do item.

In [ ]:
try:
    import duckdb
except ModuleNotFoundError:
    import piplite; await piplite.install('duckdb'); import duckdb
con = duckdb.connect()
# Staging no estilo Olist (chaves naturais da origem)
con.execute('CREATE TABLE stg_cliente(customer_id VARCHAR, estado VARCHAR)')
con.executemany('INSERT INTO stg_cliente VALUES (?,?)', [
    ('c1','SP'),('c2','RJ'),('c3','MG')])
con.execute('CREATE TABLE stg_produto(product_id VARCHAR, categoria VARCHAR)')
con.executemany('INSERT INTO stg_produto VALUES (?,?)', [
    ('p1','cama_mesa_banho'),('p2','informatica_acessorios'),('p3','moveis_decoracao')])
# 1 linha por ITEM de pedido (o grão escolhido)
con.execute('CREATE TABLE stg_item(order_id VARCHAR, customer_id VARCHAR, product_id VARCHAR, price DOUBLE, freight DOUBLE)')
con.executemany('INSERT INTO stg_item VALUES (?,?,?,?,?)', [
    ('o1','c1','p1',50.0,10.0),('o1','c1','p2',200.0,15.0),
    ('o2','c2','p1',60.0,12.0),('o3','c2','p3',300.0,40.0),
    ('o4','c3','p2',180.0,20.0),('o5','c1','p1',70.0,11.0),
    ('o6','c3','p3',250.0,35.0),('o7','c2','p2',220.0,18.0)])
con.execute('SELECT COUNT(*) AS itens FROM stg_item').df()

## Passo 3 — Dimensões com surrogate keys
Cada dimensão ganha um `sk` sequencial (ROW_NUMBER) e guarda a chave natural.

In [ ]:
con.execute('''CREATE TABLE dim_cliente AS
    SELECT ROW_NUMBER() OVER (ORDER BY customer_id) AS sk_cliente, customer_id, estado
    FROM stg_cliente''')
con.execute('''CREATE TABLE dim_produto AS
    SELECT ROW_NUMBER() OVER (ORDER BY product_id) AS sk_produto, product_id, categoria
    FROM stg_produto''')
con.execute('SELECT * FROM dim_produto ORDER BY sk_produto').df()

## Passo 4 — A fato no grão do item (surrogate key lookup)
Junta o staging às dimensões pela chave natural e guarda só as surrogate keys + métricas.

In [ ]:
con.execute('''CREATE TABLE fato_item_pedido AS
    SELECT dc.sk_cliente, dp.sk_produto, i.price, i.freight
    FROM stg_item i
    JOIN dim_cliente dc ON i.customer_id = dc.customer_id
    JOIN dim_produto dp ON i.product_id = dp.product_id''')
con.execute('SELECT * FROM fato_item_pedido').df()

## Analisando o star
Pergunta de negócio: **receita (price) por categoria por estado**.

In [ ]:
con.execute('''
    SELECT dc.estado, dp.categoria, SUM(f.price) AS receita
    FROM fato_item_pedido f
    JOIN dim_cliente dc ON f.sk_cliente = dc.sk_cliente
    JOIN dim_produto dp ON f.sk_produto = dp.sk_produto
    GROUP BY dc.estado, dp.categoria
    ORDER BY receita DESC
''').df()

## Sua vez (mini-desafio)
Traga a **receita total (price + freight) por estado**, colunas `(estado, receita)`, da maior para a menor. Verifique.

In [ ]:
resposta = con.execute('''
    SELECT dc.estado, SUM(f.price + f.freight) AS receita
    FROM fato_item_pedido f
    JOIN dim_cliente dc ON f.sk_cliente = dc.sk_cliente
    GROUP BY dc.estado
    ORDER BY receita DESC
''').fetchall()
resposta

In [ ]:
def verificar(rows):
    esperado = [('RJ',650.0),('MG',485.0),('SP',356.0)]
    try:
        assert rows == esperado, 'Some price+freight e agrupe por estado (join na dim_cliente).'
        print('✅ Correto! Você modelou e analisou o star schema do Olist.')
    except AssertionError as e:
        print('❌', e)

verificar(resposta)